In [1]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv, dotenv_values
from requests.exceptions import HTTPError, ConnectionError, Timeout, RequestException
from pathlib import Path

In [2]:
load_dotenv(Path.cwd().parent / ".env")

False

In [3]:
URL_BASE = "https://api.company-information.service.gov.uk"
URL_META_DATA = "https://document-api.company-information.service.gov.uk/document"
TIMEOUT = 10  # seconds

In [4]:
#Company House

def get(endpoint, params=None, headers=None):
    url = f"{URL_BASE}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    print(response.status_code)
    return response.json()

def get_meta_data(endpoint, params=None, headers=None):
    url = f"{URL_META_DATA}/{endpoint}"
    response = requests.get(
        url,
        auth=(os.getenv("CH_API"),""),
        headers=headers,
        params=params,
        timeout=TIMEOUT
        )
    response.raise_for_status()
    return response

In [5]:
import time, threading, itertools, re
from concurrent.futures import ThreadPoolExecutor, as_completed
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k): return x            # no-op fallback if tqdm isn't installed

# --- Multi-key throttled Companies House client (uses all API keys -> ~Nx faster)
# CH's 600-requests/5-min limit is PER API KEY. We load every CH_API* key, give
# each its own Session + pacer (~1.7 req/s), and spread requests across them via a
# thread pool. Aggregate throughput ~= N_keys * 1.7 req/s (4 keys ~= 6.7 req/s).

def _clean_key(v):
    """Extract a key value written as  KEY = 'abc',  /  "abc",  /  abc  (the .env
       here stores them Python-list style with quotes and trailing commas)."""
    v = v.strip()
    m = re.search(r"""['"]([^'"]+)['"]""", v)   # value inside the first quote pair
    return m.group(1) if m else v.rstrip(",").strip()


def _load_ch_keys(names=("CH_API", "CH_API_2", "CH_API_3", "CH_API_4")):
    """Load keys tolerantly from the repo-root .env (handles spaces around '=',
       quotes and trailing commas that python-dotenv rejects), then env vars."""
    parsed = {}
    for cand in (Path.cwd().parent.parent / ".env",
                 Path.cwd().parent / ".env",
                 Path.cwd() / ".env"):
        if cand.exists():
            for line in cand.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                parsed[k.strip()] = _clean_key(v)
            break
    keys = [parsed.get(n) or os.getenv(n) for n in names]
    return [k for k in keys if k]


_KEYS = _load_ch_keys()
if not _KEYS:
    raise RuntimeError("No Companies House API keys found (checked .env and env vars)")

_MIN_INTERVAL = 0.6           # seconds between calls PER KEY (~1.7/s; CH cap 2.0/s each)
MAX_WORKERS   = 2 * len(_KEYS)
print(f"{len(_KEYS)} API key(s) loaded -> up to ~{len(_KEYS) / _MIN_INTERVAL:.1f} req/s "
      f"with {MAX_WORKERS} workers")


class _Lane:
    """One API key: dedicated Session + a lock enforcing min-interval pacing."""
    __slots__ = ("session", "lock", "last")
    def __init__(self, key):
        self.session = requests.Session(); self.session.auth = (key, "")
        self.lock = threading.Lock(); self.last = 0.0
    def pace(self):
        with self.lock:
            wait = _MIN_INTERVAL - (time.monotonic() - self.last)
            if wait > 0:
                time.sleep(wait)
            self.last = time.monotonic()


_LANES = [_Lane(k) for k in _KEYS]
_rr = itertools.count(); _rr_lock = threading.Lock()

def _next_lane():
    with _rr_lock:
        i = next(_rr)
    return _LANES[i % len(_LANES)]


def ch_get(path, params=None, max_retries=5):
    """Thread-safe, key-rotating, throttled GET. Returns a Response (200/404/etc.
       for the caller to interpret) or None if it gave up. Handles 429 (waits out
       the window) and transient network/5xx errors with exponential backoff."""
    url = f"{URL_BASE}{path}"
    for attempt in range(max_retries):
        lane = _next_lane()
        lane.pace()
        try:
            r = lane.session.get(url, params=params, timeout=TIMEOUT)
        except (ConnectionError, Timeout):
            time.sleep(2 ** attempt); continue
        if r.status_code == 429:                       # rare with per-key pacing
            time.sleep(int(r.headers.get("Retry-After", 60)) + 1); continue
        if r.status_code >= 500:                        # transient server error
            time.sleep(2 ** attempt); continue
        return r
    return None


def parallel_fill(df, todo_index, fn, cols, out_path,
                  desc="fetch", max_workers=None, checkpoint_every=200):
    """Concurrently apply fn(com_num) over todo_index rows and write results into
       `cols` (str, or list matching fn's tuple return). Only network I/O runs in
       threads; all DataFrame writes + checkpoints happen on the main thread, so no
       df locking is needed. Resumable + checkpointed + live rate/ETA."""
    cols = [cols] if isinstance(cols, str) else list(cols)
    todo_index = list(todo_index)
    n_todo = len(todo_index)
    workers = max_workers or MAX_WORKERS
    print(f"{n_todo} to fetch (of {len(df)}) — {len(_LANES)} keys x {workers} workers")
    t0 = time.time(); done = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        fut2idx = {ex.submit(fn, df.at[idx, "com_num"]): idx for idx in todo_index}
        for fut in tqdm(as_completed(fut2idx), total=n_todo, desc=desc):
            idx = fut2idx[fut]
            try:
                res = fut.result()
            except Exception:
                res = "error" if len(cols) == 1 else tuple("error" for _ in cols)
            if len(cols) == 1:
                df.at[idx, cols[0]] = res
            else:
                for c, v in zip(cols, res):
                    df.at[idx, c] = v
            done += 1
            if done % checkpoint_every == 0:
                df.to_csv(out_path, index=False)
                rate = done / (time.time() - t0)
                eta = (n_todo - done) / rate / 60 if rate else 0
                print(f"  {done}/{n_todo} | {rate:.2f}/s | ETA ~{eta:.0f} min | checkpoint saved")
    df.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")

4 API key(s) loaded -> up to ~6.7 req/s with 8 workers


## Stage 1: profile pull — `account_type` + `accounts_overdue` (one call)

Scans **every** CSV in `company_data/company_raw/`, dedupes on `com_num`, and pulls the company profile **only for new company numbers**. One `/company/{num}` call yields **both** `account_type` and `accounts_overdue` (previously fetched in two separate passes).

Output → `company_data/company_with_acctype/com_names_with_acctype.csv`

In [6]:
# --- Stage 1: ONE profile call per NEW company -> account_type + accounts_overdue
# Scans every CSV in company_raw/, dedupes on com_num, and fetches the profile
# ONLY for company numbers not already in the output. Both fields come from the
# same /company/{num} response, so this costs no extra API calls vs account_type
# alone (previously accounts_overdue re-pulled the same profile later -- removed).
BASE        = Path("company_data")
RAW_DIR     = BASE / "company_raw"
ACCTYPE_OUT = BASE / "company_with_acctype" / "com_names_with_acctype_test.csv"

# 1. Combine every raw file; one row per company (raw files repeat com_num per SIC).
raw = pd.concat([pd.read_csv(f, dtype=str) for f in sorted(RAW_DIR.glob("*.csv"))],
                ignore_index=True).drop_duplicates("com_num")

# 2. Load prior results; NEW companies = those not already processed.
if ACCTYPE_OUT.exists():
    done = pd.read_csv(ACCTYPE_OUT, dtype=str)
else:
    done = pd.DataFrame(columns=list(raw.columns) + ["account_type"])
if "accounts_overdue" not in done.columns:
    done["accounts_overdue"] = pd.NA        # old rows: filled from the SME file later

new = raw[~raw["com_num"].isin(done["com_num"])].copy()
new["account_type"] = pd.NA
new["accounts_overdue"] = pd.NA
df = pd.concat([done, new], ignore_index=True)   # keep old results, append new rows


def fetch_profile_fields(company_num):
    """(account_type, accounts_overdue) from ONE profile call.
       Sentinels: 'no_accounts', 'not_found', 'error', 'unknown'."""
    r = ch_get(f"/company/{str(company_num).strip()}")
    if r is None:              return ("error", "error")
    if r.status_code == 404:   return ("not_found", "not_found")
    if r.status_code != 200:   return ("error", "error")
    acc = r.json().get("accounts", {})
    acctype = acc.get("last_accounts", {}).get("type") or "no_accounts"
    ov = acc.get("overdue")
    return (acctype, "unknown" if ov is None else str(ov))


# Fetch only rows missing account_type (new companies + any interrupted ones).
todo = df.index[df["account_type"].isna()]
print(f"{len(new)} new companies from raw | {len(done)} already done")
parallel_fill(df, todo, fetch_profile_fields,
              ["account_type", "accounts_overdue"], ACCTYPE_OUT, desc="profile")
df["account_type"].value_counts(dropna=False)


25171 new companies from raw | 0 already done
25171 to fetch (of 25171) — 4 keys x 8 workers
  200/25171 | 6.75/s | ETA ~62 min | checkpoint saved
  400/25171 | 6.69/s | ETA ~62 min | checkpoint saved
  600/25171 | 6.67/s | ETA ~61 min | checkpoint saved
  800/25171 | 6.65/s | ETA ~61 min | checkpoint saved
  1000/25171 | 6.65/s | ETA ~61 min | checkpoint saved
  1200/25171 | 6.65/s | ETA ~60 min | checkpoint saved
  1400/25171 | 6.64/s | ETA ~60 min | checkpoint saved
  1600/25171 | 6.64/s | ETA ~59 min | checkpoint saved
  1800/25171 | 6.64/s | ETA ~59 min | checkpoint saved
  2000/25171 | 6.64/s | ETA ~58 min | checkpoint saved
  2200/25171 | 6.63/s | ETA ~58 min | checkpoint saved
  2400/25171 | 6.63/s | ETA ~57 min | checkpoint saved
  2600/25171 | 6.63/s | ETA ~57 min | checkpoint saved
  2800/25171 | 6.62/s | ETA ~56 min | checkpoint saved
  3000/25171 | 6.62/s | ETA ~56 min | checkpoint saved
  3200/25171 | 6.62/s | ETA ~55 min | checkpoint saved
  3400/25171 | 6.62/s | ETA ~55

account_type
micro-entity                   8765
dormant                        6912
total-exemption-full           4577
no_accounts                    3767
unaudited-abridged              445
audit-exemption-subsidiary      200
full                            195
small                           160
total-exemption-small            58
group                            47
no-accounts-type-available       22
null                             12
medium                            8
audited-abridged                  2
filing-exemption-subsidiary       1
Name: count, dtype: int64

In [7]:
# --- Remove perfect duplicate rows from the acctype file -----------------------
# A "perfect duplicate" = every column identical. The pull already dedupes on
# com_num, so this is now a harmless safety net (expect 0 removed).
dedup = pd.read_csv(ACCTYPE_OUT, dtype=str)
before = len(dedup)
dedup = dedup.drop_duplicates()          # all columns must match
removed = before - len(dedup)

dedup.to_csv(ACCTYPE_OUT, index=False)
print(f"Removed {removed} perfect duplicate row(s): {before} -> {len(dedup)} rows in {ACCTYPE_OUT}")


Removed 0 perfect duplicate row(s): 25171 -> 25171 rows in company_data/company_with_acctype/com_names_with_acctype_test.csv


## Stage 2: filter to SME account types (no API calls)

Keep only companies whose `account_type` is one of `small`, `micro-entity`, `total-exemption-full`, `total-exemption-small`, `medium`. Carries forward **every** already-computed enrichment column from the existing SME file, so stage 3 only fetches genuinely new companies. Output → `com_names_sme.csv`.

In [8]:
# --- Stage 2: keep only SME-relevant account types (no API calls) ---------------
# Rebuilds the SME set from the acctype file and CARRIES FORWARD every enrichment
# column already computed in the existing SME file -- so stage 3 fetches only the
# genuinely new SME companies.
SME_TYPES = ["small", "micro-entity", "total-exemption-full",
             "total-exemption-small", "medium"]
ENRICH_COLS = ["lloyds_customer", "recent_psc_kind",
               "recent_charge_status", "recent_charge_created_on", "accounts_overdue"]

BASE        = Path("company_data")
ACCTYPE_OUT = BASE / "company_with_acctype" / "com_names_with_acctype_test.csv"
SME_OUT     = BASE / "company_sme_with_acctype" / "com_names_sme_test.csv"

full = pd.read_csv(ACCTYPE_OUT, dtype=str)
sme  = full[full["account_type"].isin(SME_TYPES)].copy()

if SME_OUT.exists():
    prev = pd.read_csv(SME_OUT, dtype=str)
    keep = ["com_num"] + [c for c in ENRICH_COLS if c in prev.columns]
    sme = sme.merge(prev[keep], on="com_num", how="left", suffixes=("", "_prev"))
    # accounts_overdue can exist in BOTH files (acctype captures it at stage 1 for
    # new companies; the SME file has it for old ones) -> prefer fresh, else prev.
    for c in ENRICH_COLS:
        if f"{c}_prev" in sme.columns:
            sme[c] = sme[c].fillna(sme[f"{c}_prev"])
            sme.drop(columns=[f"{c}_prev"], inplace=True)
for c in ENRICH_COLS:
    if c not in sme.columns:
        sme[c] = pd.NA

sme.to_csv(SME_OUT, index=False)
print(f"SME companies: {len(sme)} of {len(full)} -> {SME_OUT}")
pending = sme[ENRICH_COLS].isna().any(axis=1).sum()
print(f"  enrichment pending (stage 3) for {pending} companies")
sme["account_type"].value_counts()


SME companies: 13568 of 25171 -> company_data/company_sme_with_acctype/com_names_sme_test.csv
  enrichment pending (stage 3) for 13568 companies


account_type
micro-entity             8765
total-exemption-full     4577
small                     160
total-exemption-small      58
medium                      8
Name: count, dtype: int64

## Stage 3: SME enrichment — one charges call + one PSC call per company

For each new SME company, **2 API calls** fill **4 columns**:
- `/charges` (paged once) → `lloyds_customer` **and** `recent_charge_status` + `recent_charge_created_on` (previously two separate charges passes)
- `/persons-with-significant-control` → `recent_psc_kind`

`lloyds_customer` matches the whole Lloyds Banking Group ("Lloyds", Bank of Scotland, HBOS, Halifax, AMC, …) but **not** "Lloyd's" of London. "Most recent" = latest `created_on` / `notified_on`. Resumable — only rows missing any of the four are fetched.

In [9]:
import re

# --- Stage 3: SME enrichment (multi-key concurrent) ------------------------------
# ONE charges call (paged) + ONE PSC call per company fill FOUR columns:
#   lloyds_customer            <- charges: any charge held by a Lloyds-group entity
#   recent_charge_status       <- charges: most recent charge (latest created_on)
#   recent_charge_created_on   <- charges
#   recent_psc_kind            <- PSC: kind of the PSC with latest notified_on
# (accounts_overdue comes free with the stage-1 profile pull, not here.)
# Resumable: only processes rows still missing ANY of the four.
#
# Lloyds matching: the token "Lloyds" (no apostrophe) is the bank; it deliberately
# does NOT match "Lloyd's" (apostrophe) = Lloyd's of London, the insurance market.
# List covers the whole Lloyds Banking Group; trim to r"\blloyds\b" for bank-only.
LLOYDS_PATTERNS = [
    r"\blloyds\b",                       # Lloyds Bank, Lloyds TSB, Lloyds Commercial Finance, Lloyds Dev. Capital
    r"bank of scotland",                 # Bank of Scotland plc (Halifax's legal entity too)
    r"\bhbos\b",                         # HBOS plc
    r"agricultural mortgage corp",       # AMC — Lloyds' farm/agri lender
    r"\bhalifax\b",                      # Halifax (division of Bank of Scotland)
    r"birmingham midshires",             # BM — mortgages
    r"cheltenham (?:& |and )gloucester", # C&G — mortgages
    r"bank of wales",                    # historic BoS brand
    r"black horse",                      # Lloyds motor/asset finance
    r"scottish widows",                  # pensions/insurance
    r"\bmbna\b",                         # credit cards (Lloyds acquired 2017)
]
LLOYDS_RE = re.compile("|".join(LLOYDS_PATTERNS), re.I)

SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme_test.csv"
ENRICH_COLS = ["lloyds_customer", "recent_charge_status",
               "recent_charge_created_on", "recent_psc_kind"]

sme = pd.read_csv(SME_OUT, dtype=str)
for col in ENRICH_COLS:
    if col not in sme.columns:
        sme[col] = pd.NA


def enrich_sme(company_num):
    """(lloyds_customer, recent_charge_status, recent_charge_created_on,
        recent_psc_kind) — one paged charges call + one PSC call."""
    cnum = str(company_num).strip()

    # -- charges: single pass yields BOTH the Lloyds flag and the latest charge --
    lloyds, status, created = "False", None, None
    start, charges = 0, []
    while True:
        r = ch_get(f"/company/{cnum}/charges",
                   params={"items_per_page": 100, "start_index": start})
        if r is None:            lloyds = status = created = "error";  break
        if r.status_code == 404: status = created = "no_charges";      break
        if r.status_code != 200: lloyds = status = created = "error";  break
        data = r.json()
        items = data.get("items", [])
        charges.extend(items)
        for ch in items:
            for p in ch.get("persons_entitled", []):
                if LLOYDS_RE.search(p.get("name", "")):
                    lloyds = "True"        # keep paging: still need the latest charge
        start += len(items)
        if not items or start >= data.get("total_count", 0):
            break
    if status is None:                      # paged through without error
        if charges:
            latest = max(charges, key=lambda c: c.get("created_on") or "")
            status = latest.get("status") or "unknown"
            created = latest.get("created_on") or "unknown"
        else:
            status = created = "no_charges"

    # -- PSC: kind of the most recent PSC (latest notified_on) --
    rp = ch_get(f"/company/{cnum}/persons-with-significant-control",
                params={"items_per_page": 100})
    if rp is None:              psc = "error"
    elif rp.status_code == 404: psc = "no_psc"
    elif rp.status_code != 200: psc = "error"
    else:
        items = rp.json().get("items", [])
        if not items:
            psc = "no_psc"
        else:
            latest = max(items, key=lambda it: it.get("notified_on") or "")
            psc = latest.get("kind") or "unknown"

    return lloyds, status, created, psc


todo = sme.index[sme[ENRICH_COLS].isna().any(axis=1)]
parallel_fill(sme, todo, enrich_sme, ENRICH_COLS, SME_OUT, desc="sme_enrich")
sme[ENRICH_COLS].apply(lambda c: c.value_counts(dropna=False).head(6))


13568 to fetch (of 13568) — 4 keys x 8 workers
  200/13568 | 3.30/s | ETA ~68 min | checkpoint saved
  400/13568 | 3.30/s | ETA ~67 min | checkpoint saved
  600/13568 | 3.29/s | ETA ~66 min | checkpoint saved
  800/13568 | 3.29/s | ETA ~65 min | checkpoint saved
  1000/13568 | 3.29/s | ETA ~64 min | checkpoint saved
  1200/13568 | 3.29/s | ETA ~63 min | checkpoint saved
  1400/13568 | 3.29/s | ETA ~62 min | checkpoint saved
  1600/13568 | 3.29/s | ETA ~61 min | checkpoint saved
  1800/13568 | 3.29/s | ETA ~60 min | checkpoint saved
  2000/13568 | 3.29/s | ETA ~59 min | checkpoint saved
  2200/13568 | 3.29/s | ETA ~58 min | checkpoint saved
  2400/13568 | 3.29/s | ETA ~57 min | checkpoint saved
  2600/13568 | 3.29/s | ETA ~56 min | checkpoint saved
  2800/13568 | 3.29/s | ETA ~55 min | checkpoint saved
  3000/13568 | 3.29/s | ETA ~53 min | checkpoint saved
  3200/13568 | 3.29/s | ETA ~52 min | checkpoint saved
  3400/13568 | 3.29/s | ETA ~51 min | checkpoint saved
  3600/13568 | 3.29/s 

,lloyds_customer,recent_charge_status,recent_charge_created_on,recent_psc_kind
2022-07-29,NaN,NaN,6.0,NaN
2024-03-27,NaN,NaN,5.0,NaN
2024-12-20,NaN,NaN,8.0,NaN
2025-03-06,NaN,NaN,5.0,NaN
2026-05-13,NaN,NaN,6.0,NaN
False,13331.0,NaN,NaN,NaN
True,237.0,NaN,NaN,NaN
corporate-entity-person-with-significant-control,NaN,NaN,NaN,889.0
fully-satisfied,NaN,271.0,NaN,NaN
individual-person-with-significant-control,NaN,NaN,NaN,9824.0


## Normalise types for analysis

Load a typed view (`sme_typed`) with `lloyds_customer` as a real boolean, so filtering and `== True` work. The CSV on disk stays uniform text.

In [10]:
# --- Analysis-ready load: consistent dtypes & safe comparisons -----------------
# The CSV stores everything as text (uniform "True"/"False"/sentinels). This loads
# a typed view where the boolean flags become real nullable booleans, so you can do
# `sme_typed[sme_typed.lloyds_customer]` or `== True`. Non True/False values
# (e.g. 'error') become <NA> and are reported so nothing is hidden.
SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme_test.csv"
sme_typed = pd.read_csv(SME_OUT, dtype=str)

bool_map = {"True": True, "False": False}
for col in ("lloyds_customer", "accounts_overdue"):
    if col in sme_typed.columns:
        odd = sme_typed.loc[sme_typed[col].notna() & ~sme_typed[col].isin(bool_map), col]
        if len(odd):
            print(f"Non True/False {col} values -> <NA>:")
            print(odd.value_counts().to_string(), "\n")
        sme_typed[col] = sme_typed[col].map(bool_map).astype("boolean")
        print(f"{col}:")
        print(sme_typed[col].value_counts(dropna=False).to_string(), "\n")

for col in ("recent_psc_kind", "recent_charge_status"):
    if col in sme_typed.columns:
        print(f"{col}:")
        print(sme_typed[col].value_counts(dropna=False).to_string(), "\n")

# `sme_typed` is analysis-ready (e.g. sme_typed[sme_typed.lloyds_customer]).
# The CSV on disk stays canonical text.
sme_typed.head()


lloyds_customer:
lloyds_customer
False    13331
True       237 

accounts_overdue:
accounts_overdue
False    13253
True       315 

recent_psc_kind:
recent_psc_kind
individual-person-with-significant-control          9824
no_psc                                              2838
corporate-entity-person-with-significant-control     889
legal-person-person-with-significant-control          17 

recent_charge_status:
recent_charge_status
no_charges         11778
outstanding         1518
fully-satisfied      271
part-satisfied         1 



,com_num,name,sic_code,account_type,accounts_overdue,lloyds_customer,recent_psc_kind,recent_charge_status,recent_charge_created_on
0,15073164,NFLECTION ADVISORY LIMITED,70229,total-exemption-full,False,False,individual-person-with-significant-control,no_charges,no_charges
1,13522064,NFOGENIE LTD,58290,micro-entity,False,False,individual-person-with-significant-control,no_charges,no_charges
2,11006939,NNOV8 LIMITED,62090,micro-entity,False,False,individual-person-with-significant-control,no_charges,no_charges
3,SC606050,NSPIRED INVESTMENTS LTD,68209,total-exemption-full,False,False,individual-person-with-significant-control,outstanding,2021-09-01
4,11303802,"""1ST RATE"" PSYCHOLOGY SERVICES LTD",85600,micro-entity,False,False,individual-person-with-significant-control,no_charges,no_charges


## Repair com_num leading zeros

Realign `com_num` in `com_names_sme.csv` to the canonical values in `company_raw/` (fixes leading zeros dropped by Excel, e.g. `5914136` → `05914136`). Idempotent — run it any time, especially after opening/saving the CSV in a spreadsheet.

In [11]:
# --- Repair com_num: realign leading zeros to the raw originals ------------------
# UK company numbers keep leading zeros (e.g. 05914136). Opening the CSV in Excel
# can silently drop them (05914136 -> 5914136, or float-ify to 5914136.0). This
# remaps com_num back to the canonical values in company_raw/. Idempotent:
# reports 0 changed when the file is already aligned.
RAW_DIR = Path("company_data") / "company_raw"
SME_OUT = Path("company_data") / "company_sme_with_acctype" / "com_names_sme_test.csv"


def _norm(x):
    """Collapse a com_num to a match key: drop a stray '.0', then leading zeros."""
    x = str(x).strip()
    if x.endswith(".0"):
        x = x[:-2]
    return x.lstrip("0")


# canonical map: normalized form -> original com_num (from the raw files)
raw = pd.concat([pd.read_csv(f, dtype=str) for f in sorted(RAW_DIR.glob("*.csv"))],
                ignore_index=True).drop_duplicates("com_num")
canon = {_norm(c): c for c in raw["com_num"]}

sme = pd.read_csv(SME_OUT, dtype=str)
before = sme["com_num"].copy()
sme["com_num"] = sme["com_num"].map(lambda x: canon.get(_norm(x), str(x).strip()))

changed = int((before != sme["com_num"]).sum())
unmatched = sme.loc[~sme["com_num"].isin(set(raw["com_num"])), "com_num"]
sme.to_csv(SME_OUT, index=False)
print(f"Aligned com_num -> {SME_OUT}")
print(f"  rows changed: {changed}")
print(f"  not matching any raw com_num: {len(unmatched)}")
if len(unmatched):
    print("  examples:", unmatched.head(5).tolist())


Aligned com_num -> company_data/company_sme_with_acctype/com_names_sme_test.csv
  rows changed: 0
  not matching any raw com_num: 0
